# BRSR FinBERT Sentiment Analysis
**Project:** Greenwashing Risk Detection in Indian BRSR Disclosures Using NLP  
**Notebook role:** This notebook extends the main pipeline (`src/nlp_pipeline.py`) with FinBERT — a BERT model fine-tuned on financial text (Araci, 2019).  
**Why a separate notebook?** FinBERT requires downloading ~440 MB model weights from Hugging Face. Run this on **Google Colab (free GPU)** or any machine with internet access.

### How to use
1. Upload your `data/raw/` corpus folder, or paste BRSR text directly.
2. Run all cells in order.
3. Download `finbert_results.csv` and merge with `nlp_results.csv` from the main pipeline.

### Citation
- Araci, D. (2019). *FinBERT: Financial Sentiment Analysis with BERT.* arXiv:1908.10063.
- Devlin et al. (2019). *BERT: Pre-training of Deep Bidirectional Transformers.* NAACL-HLT. arXiv:1810.04805.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
# Runtime: ~2 min on Colab T4 GPU
!pip install transformers torch pandas numpy matplotlib seaborn nltk --quiet

In [ ]:
# ── Cell 2: Imports and device check ─────────────────────────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re, os
import nltk
from nltk.tokenize import sent_tokenize
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')  # Should print 'cuda' on Colab GPU runtime

In [ ]:
# ── Cell 3: Load FinBERT ──────────────────────────────────────────────────────
# ProsusAI/finbert is fine-tuned BERT on ~10k financial news sentences.
# Labels: 0=positive, 1=negative, 2=neutral
MODEL_NAME = 'ProsusAI/finbert'
print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model = model.to(device)
model.eval()
print('Model ready.')
# Label mapping from the model card
id2label = {0: 'positive', 1: 'negative', 2: 'neutral'}

In [ ]:
# ── Cell 4: Corpus — paste or upload your BRSR texts ─────────────────────────
# OPTION A: Upload data/raw/ folder from your GitHub repo to Colab.
#   Then set RAW_PATH to the uploaded path.
# OPTION B: Use the SAMPLE_TEXTS dict below (from real corpus extracts).

# ── SAMPLE TEXTS (subset, for reproducibility demo) ──────────────────────────
SAMPLE_TEXTS = {
    'Infosys (IT)': """
        In 2020, we became carbon neutral, 30 years ahead of the Paris Agreement timeline.
        Infosys is committed to nurturing a sustainable and socially responsible business.
        We require suppliers to accept the Supplier Code of Conduct.
        Our ESG aspirations are reflected in the Infosys ESG Vision 2030.
        R&D spend was 20.7% of capex in FY2024 directed at energy-efficient equipment.
        We aspire to be an industry leader in responsible business practices.
        100% of permanent employees are covered by health insurance.
        We aim to achieve net-zero emissions in our operations by 2040.
        Total employees: 3,40,687 as of March 31, 2024.
        Renewable electricity was 74% of our total electricity use in FY2024.
    """,
    'UltraTech Cement (Cement)': """
        The Company has set voluntary targets to reduce emissions by 27% for Scope 1
        and 69% for Scope 2 by 2032 from the 2017 base year, validated by SBTi.
        UltraTech follows a Zero Harm culture in its operations.
        Health and Safety is given great importance and the Company endeavours
        to make the workplace safe for all employees.
        The Company focuses on enhanced circular economy and 100% energy transition.
        Net Zero target is set for 2050.
        Total workforce: 83,104 employees and workers as of March 31, 2024.
        Female workforce: 6% among employees and 0.22% among workers.
        Exports account for only 0.5% of turnover.
        We strive to be a responsible corporate citizen committed to sustainability.
    """,
    "Dr. Reddy's (Pharma)": """
        Scope 1 and 2 emissions reduced by 33% versus FY2023 baseline.
        Energy-saving initiatives avoided 8,073 tonnes CO2 and saved 7.75 million kWh in FY25.
        Scope 3 emissions intensity reduced by 31% since FY2023.
        1,761 vendors confirmed alignment with Supplier Code of Conduct in FY25.
        37 vendor PSCI audits covering approximately INR 500 crore yearly spend.
        Life Cycle Assessment conducted for 2 API products.
        ESG objectives embedded in MD performance scorecard.
        Waste reduction of 20-22% achieved across 9 high-volume products.
        We are committed to affordable and innovative medicines.
        Our goal is to serve 1.5 billion patients by 2030.
    """,
    'Tata Motors (Auto)': """
        TML has set ambitious targets of achieving Net Zero GHG emissions
        by 2045 for CV business and 2040 for PV business.
        We are committed to achieving RE100 before the end of this decade.
        The Sustainable Supply Chain Initiative was launched in 2017.
        AIKYAM, a collaborative supply chain platform, was introduced in 2023.
        Turnover rate for permanent employees in FY24: 7% overall.
        Female representation on the Board: 3 out of 8 directors (37.5%).
        Exports contribute only 4% of total turnover.
        We seek to transition away from fossil fuels in our operations.
        We aim to add significant value through every regulatory norm change.
    """,
    'Cipla (Pharma)': """
        ISO 45001:2018 certified at 38 of 46 manufacturing sites globally.
        ISO 14001:2015 certified at 38 of 46 manufacturing sites globally.
        ISO 50001 energy management at 25 of 46 manufacturing sites.
        Board training: 91.44% of Board Directors covered in awareness programmes.
        Employee training coverage: 94.60% for employees, 49.64% for workers.
        Total customer complaints in FY24: 6,179 (1,111 pending resolution).
        The Company has a zero tolerance approach towards corruption and bribery.
        We aspire to be a global pharmaceutical leader in responsible practices.
        All subsidiaries participate in Business Responsibility initiatives.
        We commit to continuous improvement in our sustainability performance.
    """,
}

# ── OR: Load from file system ─────────────────────────────────────────────────
# RAW_PATH = Path('/content/data/raw')  # Update if uploading files
# if RAW_PATH.exists():
#     for fp in RAW_PATH.glob('*.txt'):
#         SAMPLE_TEXTS[fp.stem] = fp.read_text(encoding='utf-8')

print(f'Corpus loaded: {len(SAMPLE_TEXTS)} documents')

In [ ]:
# ── Cell 5: FinBERT sentence-level scoring ────────────────────────────────────

MAX_LEN = 512  # FinBERT max token length

def score_sentence(sentence: str) -> dict:
    """Return FinBERT probabilities for one sentence."""
    sentence = sentence.strip()
    if len(sentence) < 10:
        return {'positive': 0.0, 'negative': 0.0, 'neutral': 1.0, 'label': 'neutral'}
    inputs = tokenizer(
        sentence, return_tensors='pt', max_length=MAX_LEN,
        truncation=True, padding=True
    ).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = F.softmax(logits, dim=1).squeeze().cpu().numpy()
    label = id2label[probs.argmax()]
    return {'positive': round(float(probs[0]),4),
            'negative': round(float(probs[1]),4),
            'neutral':  round(float(probs[2]),4),
            'label': label}


def score_document(name: str, text: str) -> dict:
    """Score all sentences in a document and aggregate."""
    sentences = [s.strip() for s in sent_tokenize(text) if len(s.strip()) > 20]
    results = [score_sentence(s) for s in sentences]
    
    pos_count = sum(1 for r in results if r['label'] == 'positive')
    neg_count = sum(1 for r in results if r['label'] == 'negative')
    neu_count = sum(1 for r in results if r['label'] == 'neutral')
    n = max(len(results), 1)
    
    # Average probabilities
    avg_pos = np.mean([r['positive'] for r in results])
    avg_neg = np.mean([r['negative'] for r in results])
    avg_neu = np.mean([r['neutral'] for r in results])
    
    # FinBERT greenwashing signal:
    # High positive + low specificity = greenwashing risk flag
    # Here we just compute sentiment; combine with SR from main pipeline
    
    return {
        'company': name,
        'n_sentences_scored': n,
        'pct_positive': round(pos_count/n*100, 1),
        'pct_negative': round(neg_count/n*100, 1),
        'pct_neutral':  round(neu_count/n*100, 1),
        'avg_positive_prob': round(avg_pos, 4),
        'avg_negative_prob': round(avg_neg, 4),
        'avg_neutral_prob':  round(avg_neu, 4),
        'dominant_sentiment': id2label[np.argmax([avg_pos, avg_neg, avg_neu])],
        'sentence_details': list(zip(sentences, results)),
    }


# Run scoring
print('Scoring documents with FinBERT...')
doc_results = {}
for company, text in SAMPLE_TEXTS.items():
    print(f'  {company}...')
    doc_results[company] = score_document(company, text)

print('\nDone.')

In [ ]:
# ── Cell 6: Results DataFrame ─────────────────────────────────────────────────
summary_rows = [
    {k: v for k, v in r.items() if k != 'sentence_details'}
    for r in doc_results.values()
]
results_df = pd.DataFrame(summary_rows)
print(results_df[['company','n_sentences_scored','pct_positive',
                   'pct_negative','pct_neutral','dominant_sentiment']].to_string(index=False))

In [ ]:
# ── Cell 7: Sentence-level inspection (spot-check) ────────────────────────────
print('=== Sentence-level detail: Dr. Reddy\'s ===')
dr_key = [k for k in doc_results if "Reddy" in k][0]
for sent, res in doc_results[dr_key]['sentence_details']:
    label = res['label'].upper()
    conf  = max(res['positive'], res['negative'], res['neutral'])
    print(f"[{label:8s} {conf:.2f}] {sent[:90]}")

In [ ]:
# ── Cell 8: Visualisation ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stacked bar: sentiment distribution per company
companies = results_df['company'].str[:20].values
x = np.arange(len(companies))
axes[0].bar(x, results_df['pct_positive'], color='#4CAF50', label='Positive')
axes[0].bar(x, results_df['pct_negative'], bottom=results_df['pct_positive'],
            color='#F44336', label='Negative')
axes[0].bar(x, results_df['pct_neutral'],
            bottom=results_df['pct_positive'] + results_df['pct_negative'],
            color='#9E9E9E', label='Neutral')
axes[0].set_xticks(x)
axes[0].set_xticklabels(companies, rotation=35, ha='right', fontsize=8)
axes[0].set_ylabel('% of Sentences')
axes[0].set_title('FinBERT Sentiment Distribution\nper Company', fontweight='bold')
axes[0].legend(fontsize=8)

# Average probabilities heatmap
heat_data = results_df[['avg_positive_prob','avg_negative_prob','avg_neutral_prob']].values
im = axes[1].imshow(heat_data.T, cmap='RdYlGn', aspect='auto', vmin=0, vmax=0.8)
axes[1].set_xticks(range(len(companies)))
axes[1].set_xticklabels(companies, rotation=35, ha='right', fontsize=8)
axes[1].set_yticks([0,1,2])
axes[1].set_yticklabels(['Positive', 'Negative', 'Neutral'])
plt.colorbar(im, ax=axes[1], label='Avg probability')
axes[1].set_title('FinBERT Average Sentiment Probabilities\nper Company', fontweight='bold')

plt.tight_layout()
plt.savefig('finbert_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: finbert_sentiment.png')

In [ ]:
# ── Cell 9: Combine with main pipeline results ────────────────────────────────
# Download nlp_results.csv from your main pipeline outputs/ folder
# and upload it to Colab, then run:

# main_df = pd.read_csv('nlp_results.csv')
# finbert_slim = results_df[['company','pct_positive','pct_negative',
#                             'pct_neutral','dominant_sentiment']]
# merged = main_df.merge(finbert_slim, left_on='company', right_on='company', how='left')
#
# Key check: do companies with HIGH GRS (greenwashing risk) also show
# high FinBERT positive %?  If yes, that supports the greenwashing hypothesis:
# lots of positive spin, without the quantitative grounding.
#
# corr = merged[['GRS','pct_positive']].corr()
# print(corr)

# For the sample corpus:
print('Sample: GRS vs FinBERT positive% (manual from pipeline output)')
sample_check = pd.DataFrame([
    {'company': 'Infosys (IT)',         'GRS': 64.0, 'pct_positive': None},
    {'company': 'UltraTech Cement',     'GRS': 77.8, 'pct_positive': None},
    {'company': "Dr. Reddy's (Pharma)", 'GRS': 53.2, 'pct_positive': None},
    {'company': 'Tata Motors (Auto)',   'GRS': 48.9, 'pct_positive': None},
    {'company': 'Cipla (Pharma)',       'GRS': 77.6, 'pct_positive': None},
])
for i, row in sample_check.iterrows():
    key = [k for k in doc_results if row['company'].split('(')[0].strip() in k]
    if key:
        sample_check.at[i,'pct_positive'] = doc_results[key[0]]['pct_positive']
print(sample_check.to_string(index=False))

# Save results
results_df.drop(columns=['sentence_details'], errors='ignore').to_csv('finbert_results.csv', index=False)
print('\nSaved: finbert_results.csv')

In [ ]:
# ── Cell 10: Important methodological caveat ──────────────────────────────────
print("""
METHODOLOGICAL NOTES:

1. FinBERT was fine-tuned on financial NEWS sentences (earnings announcements,
   market commentary), not on corporate sustainability reports. Its labels
   (positive/negative/neutral) reflect MARKET sentiment, not truth-of-claim.
   A sentence like 'We achieved net-zero in 2020' will score POSITIVE whether
   the claim is verified or not.

2. This is why FinBERT alone cannot detect greenwashing. It must be COMBINED
   with the Specificity Ratio from the main pipeline:
      HIGH positive FinBERT + LOW SR = greenwashing signal
      HIGH positive FinBERT + HIGH SR = credible positive disclosure

3. ClimateBERT (Bingler et al., 2022) is specifically trained on climate
   disclosures and would be a stronger choice for this project. However,
   FinBERT is used here as it is publicly available, well-documented, and
   widely used in academic finance NLP.

4. For the full corpus of 15 reports, run all cells after uploading data/raw/
   and setting RAW_PATH in Cell 4 accordingly.

References:
  Araci, D. (2019). FinBERT. arXiv:1908.10063
  Bingler et al. (2022). Cheap Talk and Cherry-Picking. Finance Research Letters.
  Devlin et al. (2019). BERT. NAACL-HLT. arXiv:1810.04805
""")